In [8]:
# **SVM Model** - Fixed to handle continuous target values

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, balanced_accuracy_score
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns

# Load the prepared dataset
df = pd.read_csv(r"C:\Users\meria\Downloads\mlProject\prepared_data.csv")
df = df.sample(n=3000, random_state=42)
print(f"Dataset shape: {df.shape}")

# Vérification de la target
if 'Job_Status_num' not in df.columns:
    print("ERREUR: Colonne 'Job_Status_num' introuvable!")
    print(f"   Colonnes disponibles: {df.columns.tolist()}")
else:
    print(f"Target trouvée: Job_Status_num")

# Check Job_Status_num distribution
print("\n📊 Distribution de Job_Status_num:")
print(df['Job_Status_num'].value_counts())
print(f"Proportions:\n{df['Job_Status_num'].value_counts(normalize=True)}")

# **FIX: Convert continuous target to discrete classes**
print("\n🔄 Converting continuous target to discrete classes...")
# Since we have values around -1 and 1, we'll map them to 0 and 1
df['Job_Status_num_discrete'] = (df['Job_Status_num'] > 0).astype(int)

# Verify the new discrete encoding
job_status_mapping = {
    0: 'decreasing',
    1: 'increasing'
}
print(f"🎯 New Job_Status_num_discrete mapping: {job_status_mapping}")
print(f"New distribution:\n{df['Job_Status_num_discrete'].value_counts()}")

# Prepare target variable - USING DISCRETE VERSION
target = "Job_Status_num_discrete"

# Remove leakage features and prepare X
LEAKAGE_FEATURES = [
    'Job_Growth_Rate_%',       
    'Growth_Category',          
    'Job_Status_num',
    'Job_Status_num_discrete'  # Also remove the target from features
]

X = df.select_dtypes(include=[np.number]).copy()
X = X.drop(columns=[col for col in LEAKAGE_FEATURES if col in X.columns], errors='ignore')
y = df[target].copy()

print(f"\nTarget variable: {target}")
print(f"Features used: {X.shape[1]}")
print(f"Target distribution:\n{y.value_counts().sort_index()}")
print(f"Target mapping: {job_status_mapping}")

class_names = ['decreasing', 'increasing']
y_encoded = y.astype(int)  # Ensure it's integer type

print(f"\nClasses: {class_names}")
print(f"Class distribution: {dict(zip(class_names, np.bincount(y_encoded)))}")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, 
    test_size=0.2,
    random_state=42,
    stratify=y_encoded  # Use the discrete version for stratification
)

print(f"\n📈 Data Split:")
print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Training target distribution: {dict(zip(class_names, np.bincount(y_train)))}")

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Handle class imbalance with SMOTE
print(f"\n🔄 Class balance check:")
print(f"Before SMOTE: {dict(zip(class_names, np.bincount(y_train)))}")

# Use SMOTE to handle imbalance
smote = SMOTE(sampling_strategy='auto', random_state=42, k_neighbors=3)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
print(f"After SMOTE: {dict(zip(class_names, np.bincount(y_train_smote)))}")
X_train_final, y_train_final = X_train_smote, y_train_smote

# **SVM Baseline**
print("\n" + "="*60)
print("SVM BASELINE MODEL")
print("="*60)

svm_baseline = SVC(kernel='rbf', random_state=42)
svm_baseline.fit(X_train_scaled, y_train)
y_pred_svm_baseline = svm_baseline.predict(X_test_scaled)
acc_svm_baseline = accuracy_score(y_test, y_pred_svm_baseline)

print(f"SVM Baseline - Accuracy: {acc_svm_baseline:.4f}")
print(f"SVM Baseline - Balanced Accuracy: {balanced_accuracy_score(y_test, y_pred_svm_baseline):.4f}")

print("\nRapport de classification SVM basique :")
print(classification_report(y_test, y_pred_svm_baseline, target_names=class_names))



Dataset shape: (3000, 29)
Target trouvée: Job_Status_num

📊 Distribution de Job_Status_num:
Job_Status_num
 1.003807    1531
-0.996207    1469
Name: count, dtype: int64
Proportions:
Job_Status_num
 1.003807    0.510333
-0.996207    0.489667
Name: proportion, dtype: float64

🔄 Converting continuous target to discrete classes...
🎯 New Job_Status_num_discrete mapping: {0: 'decreasing', 1: 'increasing'}
New distribution:
Job_Status_num_discrete
1    1531
0    1469
Name: count, dtype: int64

Target variable: Job_Status_num_discrete
Features used: 25
Target distribution:
Job_Status_num_discrete
0    1469
1    1531
Name: count, dtype: int64
Target mapping: {0: 'decreasing', 1: 'increasing'}

Classes: ['decreasing', 'increasing']
Class distribution: {'decreasing': 1469, 'increasing': 1531}

📈 Data Split:
Training set: (2400, 25)
Test set: (600, 25)
Training target distribution: {'decreasing': 1175, 'increasing': 1225}

🔄 Class balance check:
Before SMOTE: {'decreasing': 1175, 'increasing': 122

In [ ]:
# **FIXED SVM Model** - Proper preprocessing and optimization

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, balanced_accuracy_score
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns

# Load and prepare data
df = pd.read_csv(r"C:\Users\meria\Downloads\mlProject\prepared_data.csv")
df = df.sample(n=3000, random_state=42)

print(f"Dataset shape: {df.shape}")

# **CRITICAL FIX: Convert continuous target to proper discrete classes**
print("🔄 Converting continuous target to proper discrete classes...")
df['Job_Status_num'] = (df['Job_Status_num'] > 0).astype(int)

# Verify the conversion
print("📊 New target distribution:")
print(df['Job_Status_num'].value_counts())
print(f"Proportions:\n{df['Job_Status_num'].value_counts(normalize=True)}")

job_status_mapping = {0: 'decreasing', 1: 'increasing'}
print(f"🎯 Target mapping: {job_status_mapping}")

# Prepare features and target
LEAKAGE_FEATURES = ['Job_Growth_Rate_%', 'Growth_Category', 'Job_Status_num']

X = df.select_dtypes(include=[np.number]).copy()
X = X.drop(columns=[col for col in LEAKAGE_FEATURES if col in X.columns], errors='ignore')
y = df['Job_Status_num'].copy()

print(f"\nFeatures used: {X.shape[1]}")
print(f"Target distribution: {y.value_counts().to_dict()}")

# Split data with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  # This ensures balanced classes in splits
)

print(f"\nData Split:")
print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Training target distribution: {y_train.value_counts().to_dict()}")

# **IMPROVED SCALING**
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nHandling class imbalance with SMOTE...")
print(f"Before SMOTE: {y_train.value_counts().to_dict()}")

# Apply SMOTE for better class balance
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
print(f"After SMOTE: {pd.Series(y_train_smote).value_counts().to_dict()}")

# **OPTIMIZED SVM WITH GRID SEARCH**
print("\n" + "="*60)
print("OPTIMIZING SVM WITH GRID SEARCH")
print("="*60)

# Define parameter grid
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.1, 0.01],
    'kernel': ['rbf', 'linear'],
    'class_weight': ['balanced', None]
}

# Use stratified K-fold for better validation
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Grid search with balanced accuracy scoring
svm_grid = GridSearchCV(
    SVC(random_state=42),
    param_grid,
    cv=cv_strategy,
    scoring='balanced_accuracy',
    n_jobs=-1,
    verbose=1
)

print("Training optimized SVM...")
svm_grid.fit(X_train_smote, y_train_smote)

# Get best model
best_svm = svm_grid.best_estimator_
print(f"✅ Best parameters: {svm_grid.best_params_}")

# **COMPREHENSIVE EVALUATION**


# Predictions
y_pred_train = best_svm.predict(X_train_scaled)
y_pred_test = best_svm.predict(X_test_scaled)

# Calculate metrics
train_accuracy = accuracy_score(y_train, y_pred_train)
test_accuracy = accuracy_score(y_test, y_pred_test)
balanced_accuracy = balanced_accuracy_score(y_test, y_pred_test)

print(f"Performance Metrics:")
print(f"  • Train Accuracy: {train_accuracy:.4f}")
print(f"  • Test Accuracy: {test_accuracy:.4f}")
print(f"  • Balanced Accuracy: {balanced_accuracy:.4f}")
print(f"  • Train-Test Gap: {abs(train_accuracy - test_accuracy):.4f}")

print("\n Detailed Classification Report:")
print(classification_report(y_test, y_pred_test, target_names=['decreasing', 'increasing']))

# **VISUALIZATIONS**

# Confusion Matrix
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
cm = confusion_matrix(y_test, y_pred_test)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['decreasing', 'increasing'], 
            yticklabels=['decreasing', 'increasing'])
plt.title('Confusion Matrix - Optimized SVM', fontweight='bold')
plt.xlabel('Predicted')
plt.ylabel('Actual')

# Feature importance (using SVM coefficients for linear kernel)
plt.subplot(1, 3, 2)
if svm_grid.best_params_['kernel'] == 'linear':
    feature_importance = pd.DataFrame({
        'feature': X.columns,
        'importance': np.abs(best_svm.coef_[0])
    }).sort_values('importance', ascending=True).tail(10)
    
    plt.barh(feature_importance['feature'], feature_importance['importance'])
    plt.title('Top 10 Feature Importances\n(Linear SVM)')
else:
    # For non-linear kernels, we can't easily get feature importance
    plt.text(0.5, 0.5, 'Feature importance\nnot available for RBF kernel', 
             ha='center', va='center', transform=plt.gca().transAxes)
    plt.title('Feature Importance\n(Not available for RBF)')

# Performance comparison
plt.subplot(1, 3, 3)
metrics = ['Train Acc', 'Test Acc', 'Balanced Acc']
values = [train_accuracy, test_accuracy, balanced_accuracy]
colors = ['lightblue', 'lightcoral', 'lightgreen']

bars = plt.bar(metrics, values, color=colors, alpha=0.8)
plt.ylim(0, 1)
plt.title('SVM Performance Metrics')
plt.ylabel('Score')

# Add value labels on bars
for bar, value in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
             f'{value:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

# **CROSS-VALIDATION RESULTS**
print("\n📊 Cross-Validation Results:")
cv_results = svm_grid.cv_results_
best_index = svm_grid.best_index_
print(f"  • Best CV Score: {cv_results['mean_test_score'][best_index]:.4f} (±{cv_results['std_test_score'][best_index]:.4f})")
print(f"  • Mean CV Score: {cv_results['mean_test_score'].mean():.4f}")

# **COMPARE WITH BASELINE**
print("\n" + "="*60)
print("COMPARISON WITH BASELINE SVM")
print("="*60)

# Baseline SVM (no optimization)
baseline_svm = SVC(kernel='rbf', random_state=42)
baseline_svm.fit(X_train_scaled, y_train)
y_pred_baseline = baseline_svm.predict(X_test_scaled)
baseline_accuracy = accuracy_score(y_test, y_pred_baseline)

print(f"Baseline SVM Accuracy: {baseline_accuracy:.4f}")
print(f"Optimized SVM Accuracy: {test_accuracy:.4f}")
print(f"Improvement: {test_accuracy - baseline_accuracy:.4f}")

if test_accuracy > baseline_accuracy:
    print("🎉 Optimization improved performance!")
else:
    print("⚠️  Optimization didn't improve performance. Consider feature engineering.")

print("\n" + "="*60)
print("SVM ANALYSIS COMPLETED!")
print("="*60)

Train Set Accuracy: 0.6795833333333333
Test Set Accuracy: 0.49666666666666665

Classification Report:

              precision    recall  f1-score   support

           0       0.48      0.40      0.44       294
           1       0.51      0.58      0.54       306

    accuracy                           0.50       600
   macro avg       0.49      0.49      0.49       600
weighted avg       0.49      0.50      0.49       600

